In [4]:
### misc
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
import time
from itertools import product

#### graphical
import matplotlib.pyplot as plt
import corner

#### ML
import sklearn
from sklearn.decomposition import PCA
import tensorflow as tf
import keras
from keras import layers

##### poke gpu
os.environ["CUDA_VISIBLE_DEVICES"]="0"

physical_devices = tf.config.list_physical_devices("GPU") 

tf.config.experimental.set_memory_growth(physical_devices[0], True)

gpu0usage = tf.config.experimental.get_memory_info("GPU:0")["current"]

print("Current GPU usage:\n"
     + " - GPU0: " + str(gpu0usage) + "B\n")

modelpath = '/home/hatte/M4/models'


Current GPU usage:
 - GPU0: 0B



In [5]:
df_full = pd.read_hdf('../grids/Chiara.hdf5', key='df') ## edit for your grid!!

#### define inputs
inputs = ['massini', 'zini', 'yini', 'alphaMLT', 'age', 'eta', 'alphaFe']

#### define outputs
classical_outputs = ['FeH', 'LPhot', 'Teff']
astero_outputs = ['numax', 'dnuSer'] # 10 modes for now

outputs = classical_outputs+astero_outputs

df = df_full[inputs+outputs]

df_norm = (df - df.min())/(df.max() - df.min())

## check df_norm.describe looks reasonable (min=0, max=1):
df_norm.describe()

#### train/test split with seed 
seed = 42

df_train = df_norm.sample(frac=0.95, random_state=seed)
df_test = df_norm.drop(df_train.index)

df_train_inputs, df_val_inputs, df_train_outputs, df_val_outputs = sklearn.model_selection.train_test_split(df_train[inputs],df_train[outputs], test_size = 0.05, random_state=seed)

print("Training set: ", len(df_train_inputs))
print("Validation set: ", len(df_val_inputs))
print("Test set: ", len(df_test))

Training set:  6754081
Validation set:  355478
Test set:  374187


In [6]:
n_layers = [2,4,6,8]
units = [32,64,128,256,512]
loss_func = ['MSE']
Nepochs = 200
learning_rate = 0.001
arch_df = pd.DataFrame(product(n_layers,units,loss_func))

In [ ]:
for i in arch_df.index:
    ######## define architecture:
    model_name = 'simple_network'
    n_dense_layers = arch_df.loc[i, 0] #number of dense layers
    dense_layer_units = arch_df.loc[i, 1] #neurons per dense layer
    loss_func = arch_df.loc[i, 2]

    ###### Checkpointing

    checkpoint_dir = f'{modelpath}/short-run-tests/checkpoint/chk-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.model.keras'

    historyfile = f'short-run-tests/history/hist-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.json'
    
    cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1)

    ######## map out model architecture
    #### input layer
    nn_input = keras.Input(shape=(len(inputs),))

    #### dense layer(s)
    for n_dense_layer in range(n_dense_layers):
        if n_dense_layer == 0:
            dense_layer = layers.Dense(dense_layer_units, activation='relu')(nn_input)
        else:
            dense_layer = layers.Dense(dense_layer_units, activation='relu')(dense_layer)

    #### output layer
    nn_output =  layers.Dense(len(outputs), activation='linear')(dense_layer)

    ######## store architecture as keras model
    model = keras.Model(inputs=nn_input, outputs=nn_output, name=model_name)

    tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'../logs/short-run-tests-2/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

    model.compile(loss=loss_func, optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))

    history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=10000, #change higher
          verbose=1,
          epochs=Nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback]) 


    hist_df = pd.DataFrame(history.history)
    
    with open(os.path.join(modelpath, historyfile), mode="w") as f:
        hist_df.to_json(f)



Epoch 1/200


I0000 00:00:1734017898.321071  875941 service.cc:148] XLA service 0x7981e80029a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1734017898.321099  875941 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4500, Compute Capability 8.6
2024-12-12 15:38:18.351133: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1734017898.412516  875941 cuda_dnn.cc:529] Loaded cuDNN version 90300
2024-12-12 15:38:18.454318: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2024-12-12 15:38:18.92917

 91/676 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0797

I0000 00:00:1734017900.005756  875941 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


642/676 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0365

...

664/676 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0201

2024-12-12 15:56:51.626913: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_323', 4 bytes spill stores, 4 bytes spill loads

2024-12-12 15:56:51.736173: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_321', 36 bytes spill stores, 48 bytes spill loads

2024-12-12 15:56:51.859546: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_323', 284 bytes spill stores, 288 bytes spill loads

2024-12-12 15:56:51.908612: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_164_0', 756 bytes spill stores, 444 bytes spill loads

2024-12-12 15:56:51.910842: I external/local_xla/xla/stream_

676/676 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0200

2024-12-12 15:56:53.538281: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 204 bytes spill stores, 204 bytes spill loads

2024-12-12 15:56:53.699842: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 744 bytes spill stores, 440 bytes spill loads




Epoch 1: saving model to /home/hatte/M4/models/short-run-tests/checkpoint/chk-simple_network-nlayers-2-nunits-512-epochs-200-lrate-0.001-lossfunc-MSE.model.keras
676/676 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.0200 - val_loss: 0.0141
Epoch 2/200
661/676 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0137
Epoch 2: saving model to /home/hatte/M4/models/short-run-tests/checkpoint/chk-simple_network-nlayers-2-nunits-512-epochs-200-lrate-0.001-lossfunc-MSE.model.keras
676/676 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0136 - val_loss: 0.0111
Epoch 3/200
675/676 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0111
Epoch 3: saving model to /home/hatte/M4/models/short-run-tests/checkpoint/chk-simple_network-nlayers-2-nunits-512-epochs-200-lrate-0.001-lossfunc-MSE.model.keras
676/676 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0111 - val_loss: 0.0085
Epoch 4/200
670/676 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0090
Epoch 4: saving model to /home/hatte/M4/models/short-run-tests/checkpoint/chk-simple